# Raw region inference
Run after preparing frozen frames, SAM regions and automatic references. Only RGB and SAM masks enter inference. This notebook calls the reusable package and keeps downloads and GPU work opt-in. See `docs/inference.md` for dependencies and resume behavior. GPU integration and the 6 GB target remain unmeasured.

In [ ]:
from pathlib import Path
import json
import os
import sys

# Change these roots or set the environment variables; all platforms are supported.
configured_repo = os.environ.get('TRAVERSABILITY_REPO_ROOT')
if configured_repo:
    candidates = [Path(configured_repo).expanduser().resolve()]
else:
    working_dir = Path.cwd().resolve()
    candidates = [working_dir, *working_dir.parents]
candidates = [path for base in candidates for path in (base, base / 'VLM_evaluation')]
REPOSITORY_ROOT = next((p for p in candidates
                        if (p / 'src/traversability_inference').is_dir()), None)
if REPOSITORY_ROOT is None or not (REPOSITORY_ROOT / 'src/traversability_inference').is_dir():
    raise RuntimeError('Set TRAVERSABILITY_REPO_ROOT to VLM_evaluation or its repository checkout.')

def configured_root(variable, default):
    value = Path(os.environ.get(variable, default)).expanduser()
    return (value if value.is_absolute() else REPOSITORY_ROOT / value).resolve()

DATA_ROOT = configured_root('TRAVERSABILITY_DATA_ROOT', 'data')
RUN_ROOT = configured_root('TRAVERSABILITY_RUN_ROOT', 'runs')
CACHE_ROOT = configured_root('TRAVERSABILITY_CACHE_ROOT', '.cache')
RUN_NAME = 'rellis_material_v1'  # existing prepared run
RUN_DIR = RUN_ROOT / RUN_NAME
MODEL_KEY = 'qwen3_vl_4b'  # qwen3_5_4b, clip_vit_b32, or explicit qwen3_5_2b fallback
REGION_IDS = None  # None uses the frozen selected subset

# Enable each expensive step explicitly, then rerun that cell.
DOWNLOAD_WEIGHTS = False
CALIBRATE_DEVELOPMENT = False
LOAD_MODEL = False
RUN_INFERENCE = False

sys.path.insert(0, str(REPOSITORY_ROOT / 'src'))
from traversability_inference import load_backend, run_inference, calibrate_clip
from traversability_inference.configuration import MODEL_SPECS, normalize_settings

settings = json.loads((REPOSITORY_ROOT / 'configs/inference/default.json').read_text())
settings.update(cache_dir=str(CACHE_ROOT / 'huggingface/hub'), local_files_only=True)
if MODEL_KEY == 'clip_vit_b32':
    settings['calibration_path'] = str(RUN_DIR / 'metadata/clip_calibration.json')
settings = normalize_settings(MODEL_KEY, settings)
if 'backend' not in globals():
    backend = None  # retain ownership when rerunning this configuration cell
print({'repository': str(REPOSITORY_ROOT), 'data': str(DATA_ROOT),
       'run': str(RUN_DIR), 'cache': str(CACHE_ROOT), 'model': MODEL_KEY})

## Check frozen inputs
This step reads only frame and region manifests. Missing annotations are relevant only to CLIP development calibration; own phone clips are optional. No reference masks or class fractions are used by inference.

In [ ]:
from traversability_inference.storage import read_manifests

if not (RUN_DIR / 'frames.jsonl').is_file() or not (RUN_DIR / 'regions.jsonl').is_file():
    print('Prepare frames.jsonl and regions.jsonl in RUN_DIR using the data workflow before inference.')
    frames, regions = [], []
else:
    frames, regions = read_manifests(RUN_DIR)
    print({'frames': len(frames), 'selected_regions': sum(r['selected_for_classification'] for r in regions)})

## Explicit checkpoint download
This downloads only the selected pinned checkpoint into the configured cache. It does not load a model or launch a GPU workload. Install `requirements/inference.txt` first.

In [ ]:
if DOWNLOAD_WEIGHTS:
    from huggingface_hub import snapshot_download
    spec = MODEL_SPECS[MODEL_KEY]
    snapshot_download(repo_id=spec['checkpoint'], revision=spec['revision'],
                      cache_dir=settings['cache_dir'],
                      allow_patterns=['*.json', '*.jinja', '*.safetensors', '*.txt', '*.model', '*.tiktoken'])
else:
    print('Download disabled. Set DOWNLOAD_WEIGHTS=True to fetch the selected pinned checkpoint.')

## Explicit CLIP development calibration
Run only for CLIP, before classification loading. The helper loads one scoring model and closes it afterward. It needs completed valid development annotations for both reference classes under the selected robot profile. Test references never fit the threshold. The outcome may explicitly reject all targets; zero observed development unsafe acceptance gives no held-out safety guarantee.

In [ ]:
if CALIBRATE_DEVELOPMENT:
    if MODEL_KEY != 'clip_vit_b32':
        raise RuntimeError('Development calibration applies only to clip_vit_b32.')
    if backend is not None:
        raise RuntimeError('Run the cleanup cell before loading the calibration scorer.')
    if not (RUN_DIR / 'annotations.jsonl').is_file():
        raise RuntimeError('Generate development automatic references before CLIP calibration.')
    calibration = calibrate_clip(RUN_DIR, settings, DATA_ROOT)
    print({'threshold': calibration['threshold'], 'reject_all': calibration['reject_all'],
           'observed_counts': calibration['observed_counts']})
else:
    print('Development calibration disabled. Enable CALIBRATE_DEVELOPMENT for a CLIP run.')

## Explicit model loading
The adapters require one selected CUDA device, cached weights and the pinned dependency stack. CLIP also requires a compatible calibration file. If loading fails, inspect the dependency/device error; change MODEL_KEY explicitly to request the 2B fallback. Check shared-server resource availability and local server rules before any GPU job.

In [ ]:
if LOAD_MODEL:
    if backend is not None:
        raise RuntimeError('Run the cleanup cell before loading another model.')
    backend = load_backend(MODEL_KEY, settings)
    print({'model': MODEL_KEY, 'device': settings['device'], 'loaded': True})
else:
    print('Loading disabled. Set LOAD_MODEL=True after dependencies, cache and GPU are ready.')

## Resumable raw inference
The runner preserves IDs/timestamps through untouched manifests, publishes immutable metadata first and saves predictions atomically after each frame. Compatible subsets and errors survive resume. Configuration or input changes require a fresh run directory. Inference reads no annotation file. Model loading is separate from the benchmark's timed `predict_frame` method.

In [ ]:
if RUN_INFERENCE:
    if backend is None:
        raise RuntimeError('Enable and run the model-loading step first.')
    try:
        predictions = run_inference(RUN_DIR, MODEL_KEY, settings, DATA_ROOT,
                                    backend=backend, region_ids=REGION_IDS)
    except BaseException:
        backend.close()
        backend = None
        raise
    print({'saved_predictions': len(predictions),
           'errors': sum(p['status'] == 'error' for p in predictions),
           'output': str(RUN_DIR / 'predictions' / f'{MODEL_KEY}.jsonl')})
else:
    print('Inference disabled. Set RUN_INFERENCE=True when the prepared run and model are ready.')

## Cleanup
Run after inference and before another model or profiling. Cleanup is safe to repeat. Classification/evaluation and measured deployment profiling belong to their separate notebooks.

In [ ]:
if backend is not None:
    backend.close()
    backend = None
print('Inference backend released.')